# Flow–Speed–Headway Mechanism CheckPurpose: establish *why* traffic flow correlates **positively** with time headway(rho = +0.176), i.e. test the speed-mediated pathway  `Flow -> (lower) Speed -> (longer) Headway`.**Critical confound:** the two sites have non-overlapping flow ranges(Tikatuli 5145-7971, Shahjahanpur 1896-2744 pcu/h). Pooled flow effects are thereforepartly site effects. Cells 5-6 separate them by testing flow *within* each site.Run cells in order. Cell 2 must be edited to point at your data file.

## Cell 1 — Imports, paths, plotting style

In [ ]:
import os, warnings, itertools
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from scipy import stats

warnings.filterwarnings("ignore")

BASE     = r"D:\Headway"
GRAPHICS = os.path.join(BASE, "Graphics")
TABLES   = os.path.join(BASE, "Tables")
os.makedirs(GRAPHICS, exist_ok=True)
os.makedirs(TABLES,   exist_ok=True)

# --- plotting style to match existing manuscript figures ---
mpl.rcParams.update({
    "font.family"      : "serif",
    "font.serif"       : ["Times New Roman", "DejaVu Serif"],
    "font.size"        : 11,
    "axes.titlesize"   : 12,
    "axes.labelsize"   : 11,
    "axes.spines.top"  : False,
    "axes.spines.right": False,
    "legend.frameon"   : False,
    "figure.dpi"       : 110,
    "savefig.dpi"      : 600,
    "savefig.bbox"     : "tight",
})

SEED = 42
rng  = np.random.default_rng(SEED)
print("Base folder :", BASE)
print("Graphics    :", GRAPHICS)
print("Tables      :", TABLES)

## Cell 2 — Load data and map column names**Edit `DATA_FILE` and, if needed, the `COLS` dictionary.**The script prints all columns it finds so you can correct any mismatch before proceeding.

In [ ]:
# ---------------- EDIT THIS ----------------
DATA_FILE = os.path.join(BASE, "headway_dataset.xlsx")   # <-- your extracted dataset
SHEET     = 0                                            # sheet name/index if Excel
# -------------------------------------------

if DATA_FILE.lower().endswith((".xlsx", ".xls")):
    df = pd.read_excel(DATA_FILE, sheet_name=SHEET)
else:
    df = pd.read_csv(DATA_FILE)

print("Shape:", df.shape)
print("\nColumns found:")
for c in df.columns:
    print("   ", repr(c))

# ---- map your column names to the canonical names used below ----
COLS = {
    "TH"      : "Time_Headway_s",        # time headway (s)
    "VT"      : "Target_Speed_km/hr",    # target vehicle speed
    "VL"      : "Leading_Speed_km/hr",   # leading vehicle speed
    "DV"      : "Speed_Difference",      # target - leader
    "FLOW"    : "Flow_pcu/hr",           # traffic flow
    "SITE"    : "Site",
    "TARGET"  : "V_Target",              # BTW / PR
    "LEADCLS" : "V_Leading_Class",
}

# tolerant auto-match if an exact name is missing
def _resolve(name, cols):
    if name in cols: return name
    key = name.lower().replace(" ", "").replace("_", "").replace("/", "")
    for c in cols:
        if c.lower().replace(" ", "").replace("_", "").replace("/", "") == key:
            return c
    return None

missing = []
for k, v in list(COLS.items()):
    r = _resolve(v, df.columns)
    if r is None:
        missing.append((k, v))
    else:
        COLS[k] = r

if missing:
    print("\n!! UNRESOLVED COLUMNS — edit COLS above:", missing)
else:
    print("\nAll columns resolved:")
    for k, v in COLS.items(): print(f"   {k:8s} -> {v}")

## Cell 3 — Clean, coerce types, and summarise the flow/site overlap problem

In [ ]:
d = df.copy()

num_keys = ["TH", "VT", "VL", "DV", "FLOW"]
for k in num_keys:
    d[COLS[k]] = pd.to_numeric(d[COLS[k]], errors="coerce")

d = d.dropna(subset=[COLS[k] for k in num_keys]).reset_index(drop=True)
print("Rows after dropping missing numerics:", len(d))

TH, VT, VL, DV, FLOW = (COLS[k] for k in num_keys)
SITE, TARGET, LEADCLS = COLS["SITE"], COLS["TARGET"], COLS["LEADCLS"]

# --- flow range by site: quantifies the collinearity ---
site_flow = (d.groupby(SITE)[FLOW]
               .agg(n="size", mean="mean", sd="std", min="min", max="max")
               .round(2))
print("\nFlow (pcu/h) by site:\n", site_flow)

lo = site_flow["max"].min(); hi = site_flow["min"].max()
overlap = max(0.0, lo - hi)
print(f"\nOverlapping flow range across sites: {overlap:.1f} pcu/h "
      f"({'NO OVERLAP - flow is nearly a site proxy' if overlap <= 0 else 'partial overlap'})")

# point-biserial style check: how well does flow separate the sites?
site_codes = pd.factorize(d[SITE])[0]
rb, prb = stats.pointbiserialr(site_codes, d[FLOW])
print(f"Flow vs Site point-biserial r = {rb:.3f} (p = {prb:.3g})")

## Cell 4 — Helper functions: Spearman with CI, and partial SpearmanPartial Spearman is computed as a partial Pearson correlation on rank-transformedvariables; p-values from the t-distribution with df = n - 2 - (number of controls).

In [ ]:
def spearman_ci(x, y, n_boot=5000, seed=SEED):
    """Spearman rho with bootstrap 95% CI."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    m = ~(np.isnan(x) | np.isnan(y))
    x, y = x[m], y[m]
    rho, p = stats.spearmanr(x, y)
    r = np.random.default_rng(seed)
    idx = r.integers(0, len(x), size=(n_boot, len(x)))
    boots = np.array([stats.spearmanr(x[i], y[i]).statistic for i in idx])
    lo, hi = np.nanpercentile(boots, [2.5, 97.5])
    return dict(n=len(x), rho=rho, p=p, ci_lo=lo, ci_hi=hi)

def partial_spearman(data, x, y, controls):
    """Spearman correlation of x,y controlling for `controls` (list of column names)."""
    cols = [x, y] + list(controls)
    sub  = data[cols].dropna()
    R    = sub.rank()
    n, k = len(sub), len(controls)

    Xc = np.column_stack([np.ones(n)] + [R[c].values for c in controls])
    def resid(v):
        beta, *_ = np.linalg.lstsq(Xc, v, rcond=None)
        return v - Xc @ beta

    rx, ry = resid(R[x].values), resid(R[y].values)
    r = np.corrcoef(rx, ry)[0, 1]
    dfree = n - 2 - k
    t = r * np.sqrt(dfree / max(1e-12, 1 - r**2))
    p = 2 * stats.t.sf(abs(t), dfree)
    return dict(n=n, partial_rho=r, p=p, df=dfree, controls=", ".join(controls))

def magnitude(r):
    a = abs(r)
    return ("negligible" if a < 0.10 else "small" if a < 0.30
            else "medium" if a < 0.50 else "large")

print("Helpers ready.")

## Cell 5 — Zero-order correlations: pooled, by site, and by target type

In [ ]:
pairs = [(FLOW, VT), (FLOW, VL), (FLOW, TH), (VT, TH), (VL, TH), (DV, TH), (VT, VL)]

rows = []
def add(scope, label, sub):
    for a, b in pairs:
        if len(sub) < 10: continue
        res = spearman_ci(sub[a], sub[b])
        rows.append({"Scope": scope, "Subset": label, "X": a, "Y": b,
                     "n": res["n"], "rho": round(res["rho"], 4),
                     "CI_low": round(res["ci_lo"], 4), "CI_high": round(res["ci_hi"], 4),
                     "p": res["p"], "Magnitude": magnitude(res["rho"])})

add("Pooled", "All", d)
for s, sub in d.groupby(SITE):   add("By site",   str(s), sub)
for t, sub in d.groupby(TARGET): add("By target", str(t), sub)
for (s, t), sub in d.groupby([SITE, TARGET]):
    add("Site x target", f"{s} | {t}", sub)

corr_tbl = pd.DataFrame(rows)
corr_tbl["p"] = corr_tbl["p"].apply(lambda v: f"{v:.3g}")

print("=== KEY RESULT: Flow vs Target speed ===")
print(corr_tbl[(corr_tbl.X == FLOW) & (corr_tbl.Y == VT)].to_string(index=False))
print("\n=== Flow vs Time headway ===")
print(corr_tbl[(corr_tbl.X == FLOW) & (corr_tbl.Y == TH)].to_string(index=False))

## Cell 6 — Partial correlations: does speed explain away the flow effect?Three tests:1. `Flow ~ Headway | Target speed` — if rho collapses toward zero, the flow effect is speed-mediated.2. `Flow ~ Headway | Site` — if rho collapses, the "flow" effect was really a site effect.3. `Flow ~ Headway | Site + Target speed` — both controls together.

In [ ]:
partial_rows = []
def padd(scope, sub, controls, tag):
    if len(sub) < 15 + len(controls): return
    res = partial_spearman(sub, FLOW, TH, controls)
    zero = stats.spearmanr(sub[FLOW], sub[TH])
    partial_rows.append({
        "Scope": scope, "Control set": tag, "n": res["n"],
        "rho_zero_order": round(zero.statistic, 4),
        "rho_partial": round(res["partial_rho"], 4),
        "Attenuation_%": round(100 * (1 - abs(res["partial_rho"]) /
                                       max(1e-9, abs(zero.statistic))), 1),
        "p_partial": f"{res['p']:.3g}",
        "Magnitude": magnitude(res["partial_rho"])})

site_dummy = pd.Series(pd.factorize(d[SITE])[0], index=d.index, name="_site_code")
d2 = d.assign(_site_code=site_dummy)

padd("Pooled", d2, [VT],               "Target speed")
padd("Pooled", d2, ["_site_code"],     "Site")
padd("Pooled", d2, ["_site_code", VT], "Site + Target speed")
for s, sub in d2.groupby(SITE):
    padd(f"Within {s}", sub, [VT], "Target speed")

partial_tbl = pd.DataFrame(partial_rows)
print(partial_tbl.to_string(index=False))
print("""
INTERPRETATION GUIDE
  Large attenuation when controlling Target speed  -> flow acts on headway THROUGH speed (mechanism confirmed).
  Large attenuation when controlling Site          -> the flow effect is largely a between-site artefact.
  Flow-headway stays significant WITHIN each site  -> genuine flow effect, not merely site.
""")

## Cell 7 — Bootstrap mediation: Flow → Target speed → Time headway (on ranks)

In [ ]:
def rank_mediation(sub, n_boot=5000, seed=SEED):
    s = sub[[FLOW, VT, TH]].dropna()
    n = len(s)
    R = s.rank()
    Xf, Xm, Yt = R[FLOW].values, R[VT].values, R[TH].values

    def fit(Xcols, y):
        X = np.column_stack([np.ones(len(y))] + Xcols)
        beta, *_ = np.linalg.lstsq(X, y, rcond=None)
        return beta

    def indirect(Xf, Xm, Yt):
        a = fit([Xf], Xm)[1]                 # flow -> speed
        b = fit([Xf, Xm], Yt)[2]             # speed -> headway | flow
        c  = fit([Xf], Yt)[1]                # total
        cp = fit([Xf, Xm], Yt)[1]            # direct
        return a, b, a*b, c, cp

    a, b, ab, c, cp = indirect(Xf, Xm, Yt)
    r = np.random.default_rng(seed)
    idx = r.integers(0, n, size=(n_boot, n))
    boots = np.array([indirect(Xf[i], Xm[i], Yt[i])[2] for i in idx])
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return dict(n=n, a_flow_to_speed=a, b_speed_to_TH=b, total_c=c, direct_c_prime=cp,
                indirect_ab=ab, ind_CI_low=lo, ind_CI_high=hi,
                prop_mediated=ab / c if abs(c) > 1e-12 else np.nan)

med_rows = []
m = rank_mediation(d); m["Scope"] = "Pooled"; med_rows.append(m)
for s, sub in d.groupby(SITE):
    if len(sub) > 40:
        mm = rank_mediation(sub); mm["Scope"] = f"Within {s}"; med_rows.append(mm)

med_tbl = pd.DataFrame(med_rows)[["Scope","n","a_flow_to_speed","b_speed_to_TH",
                                  "total_c","direct_c_prime","indirect_ab",
                                  "ind_CI_low","ind_CI_high","prop_mediated"]].round(4)
print(med_tbl.to_string(index=False))
print("\nIndirect effect is significant if the 95% CI excludes zero.")

## Cell 8 — Binned flow profile (for reporting mean speed and headway across flow levels)

In [ ]:
bin_rows = []
for scope, sub in [("Pooled", d)] + [(str(s), g) for s, g in d.groupby(SITE)]:
    if len(sub) < 40: continue
    q = pd.qcut(sub[FLOW], q=min(5, sub[FLOW].nunique()), duplicates="drop")
    agg = (sub.assign(_bin=q)
              .groupby("_bin")
              .agg(n=(TH, "size"),
                   Flow_mean=(FLOW, "mean"),
                   Target_speed_mean=(VT, "mean"),
                   Target_speed_sd=(VT, "std"),
                   Headway_mean=(TH, "mean"),
                   Headway_sd=(TH, "std"))
              .reset_index())
    agg.insert(0, "Scope", scope)
    agg["_bin"] = agg["_bin"].astype(str)
    bin_rows.append(agg)

bin_tbl = pd.concat(bin_rows, ignore_index=True).round(3)
print(bin_tbl.to_string(index=False))

## Cell 9 — Figure: flow vs speed, flow vs headway, speed vs headway

In [ ]:
COLORS = {}
sites = sorted(d[SITE].unique())
palette = ["#1f77b4", "#d62728", "#2ca02c", "#9467bd"]
for i, s in enumerate(sites):
    COLORS[s] = palette[i % len(palette)]

def panel(ax, xcol, ycol, xlabel, ylabel, title):
    for s in sites:
        sub = d[d[SITE] == s]
        ax.scatter(sub[xcol], sub[ycol], s=14, alpha=0.45,
                   color=COLORS[s], edgecolors="none", label=str(s))
        if len(sub) > 20 and sub[xcol].nunique() > 5:
            z = np.polyfit(sub[xcol], sub[ycol], 1)
            xs = np.linspace(sub[xcol].min(), sub[xcol].max(), 50)
            ax.plot(xs, np.polyval(z, xs), color=COLORS[s], lw=1.8)
    rho, p = stats.spearmanr(d[xcol], d[ycol])
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel); ax.set_title(title)
    ax.text(0.03, 0.95, f"pooled $\\rho$ = {rho:.3f}\n$p$ = {p:.3g}",
            transform=ax.transAxes, va="top", ha="left", fontsize=9)

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.2))
panel(axes[0], FLOW, VT, "Traffic flow (pcu/h)", "Target speed (km/h)",
      "(a) Flow vs target speed")
panel(axes[1], FLOW, TH, "Traffic flow (pcu/h)", "Time headway (s)",
      "(b) Flow vs time headway")
panel(axes[2], VT, TH, "Target speed (km/h)", "Time headway (s)",
      "(c) Target speed vs time headway")
axes[0].legend(title=None, loc="upper right", fontsize=9)
fig.tight_layout()

out = os.path.join(GRAPHICS, "Fig_Flow_Speed_Headway_mechanism.png")
fig.savefig(out)
print("Saved:", out)
plt.show()

## Cell 10 — Export all tables to a single Excel workbook

In [ ]:
xlsx_path = os.path.join(TABLES, "Flow_Speed_Headway_mechanism.xlsx")
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as xw:
    site_flow.reset_index().to_excel(xw, sheet_name="Flow_by_site",       index=False)
    corr_tbl.to_excel(xw,             sheet_name="Zero_order_corr",       index=False)
    partial_tbl.to_excel(xw,          sheet_name="Partial_corr",          index=False)
    med_tbl.to_excel(xw,              sheet_name="Mediation",             index=False)
    bin_tbl.to_excel(xw,              sheet_name="Flow_bins",             index=False)

print("Saved:", xlsx_path)

# --- compact console summary for writing up ---
fs_pool = spearman_ci(d[FLOW], d[VT])
print(f"\nPooled flow vs target speed: rho = {fs_pool['rho']:.3f} "
      f"[{fs_pool['ci_lo']:.3f}, {fs_pool['ci_hi']:.3f}], p = {fs_pool['p']:.3g}")
for s, sub in d.groupby(SITE):
    r = spearman_ci(sub[FLOW], sub[VT])
    print(f"  within {s}: rho = {r['rho']:.3f} "
          f"[{r['ci_lo']:.3f}, {r['ci_hi']:.3f}], p = {r['p']:.3g}, n = {r['n']}")